# LensaRak — Train YOLOv8 on Google Colab (Free GPU)

**PENTING — versi ni auto-save terus ke Google Drive kau sepanjang training** (checkpoint setiap 5 epoch + `best.pt` akhir), supaya kalau runtime disconnect lagi (inactivity/max duration), hasil TAK hilang — tinggal buka Google Drive untuk ambil balik.

**Cara guna:**
1. Klik **Runtime → Change runtime type → Hardware accelerator → GPU (T4)** → Save
2. Run setiap cell ni turun bawah satu-satu (Shift+Enter)
3. Cell 3 akan minta kau **authorize Google Drive** — klik allow (dataset & model akan save dalam folder `MyDrive/LensaRak_training/`)
4. Lepas training siap, cell terakhir download `best.pt` terus ke laptop — tapi walau kau tak sempat run cell terakhir tu (disconnect dulu), fail dah selamat kat Google Drive kau, tinggal download manual dari drive.google.com


## 1. Check GPU (pastikan bukan CPU)

In [ ]:
!nvidia-smi

Kalau keluar error "command not found" — pergi balik **Runtime → Change runtime type → GPU** dulu, lepas tu run cell ni semula.

## 2. Install package

In [ ]:
!pip install -q roboflow ultralytics

## 3. Mount Google Drive (untuk auto-backup — WAJIB run cell ni)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/LensaRak_training"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Backup folder ready:", DRIVE_DIR)

## 4. Download dataset dari Roboflow

(Value ni sama macam yang kau guna kat laptop — api_key, workspace, project, version)

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="uadYEt7wU0QZkdOwz9S9")
project = rf.workspace("syaman-msa-gmail-com").project("lensarak-latest-final")
version = project.version(1)
dataset = version.download("yolov8")
print("Dataset ready at:", dataset.location)

## 5. Train (GPU — jauh lebih laju dari laptop CPU)

`project=DRIVE_DIR` bawah ni bermakna checkpoint (`last.pt`) dan `best.pt` di-save TERUS dalam Google Drive kau setiap epoch (`save_period=5` = checkpoint disimpan setiap 5 epoch) — bukan dalam storan sementara Colab. Kalau disconnect tengah-tengah, tinggal reconnect dan continue, atau ambil terus checkpoint terakhir dari Drive.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    project=DRIVE_DIR,
    name="lensarak_run",
    save_period=5,
    exist_ok=True,
)

print("\nClasses this model knows:", model.names)
print("Saved in Google Drive at:", results.save_dir)

## 6. Download `best.pt` terus ke laptop kau

Lepas ni siap, letak `best.pt` dalam folder `LensaRak_Final` (replace yang lama).

**Kalau session disconnect sebelum sempat run cell ni:** buka **drive.google.com** → folder **`LensaRak_training/lensarak_run/weights/`** → download `best.pt` dari situ terus (tak perlu run apa-apa kat Colab lagi).

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

trained_weights = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(trained_weights, "best.pt")
files.download("best.pt")